# Silver Layer Transformation

This notebook transforms the seven raw Bronze tables into typed, deduplicated Silver products for the AgentOps Control Tower.

It creates:
- **silver_costs** - normalized FOCUS cost records.
- **silver_operations** - conformed requests, dependencies, and audit events.
- **silver_agent_metrics** - custom agent metrics at a 5-minute grain.
- **silver_platform_metrics** - Azure platform metrics at a 5-minute grain.
- **silver_resources** - normalized Azure Resource Graph inventory and governance tags.

The platform-metric source belongs to the Log Analytics workspace. It is not Fabric capacity-utilization telemetry, so this notebook does not relabel it as capacity data.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, MapType, StructType


def source_column(df, *names, data_type="string"):
    columns_by_name = {column.lower(): column for column in df.columns}
    expressions = [
        F.col(f"`{columns_by_name[name.lower()]}`").cast(data_type)
        for name in names
        if name.lower() in columns_by_name
    ]
    return F.coalesce(*expressions) if expressions else F.lit(None).cast(data_type)


def source_json_column(df, *names):
    fields_by_name = {field.name.lower(): field for field in df.schema.fields}
    for name in names:
        field = fields_by_name.get(name.lower())
        if field:
            value = F.col(f"`{field.name}`")
            if isinstance(field.dataType, (ArrayType, MapType, StructType)):
                return F.to_json(value)
            return value.cast("string")
    return F.lit(None).cast("string")


## 1. Read Bronze Delta Tables

Load the seven source-aligned Delta tables produced by Notebook 1. Each table retains its original source schema and lineage columns.


In [ ]:
bronze_tables = {
    "costs": spark.read.format("delta").load("Tables/dbo/bronze_costs"),
    "resource_metadata": spark.read.format("delta").load("Tables/dbo/bronze_resource_metadata"),
    "apprequests": spark.read.format("delta").load("Tables/dbo/bronze_apprequests"),
    "appdependencies": spark.read.format("delta").load("Tables/dbo/bronze_appdependencies"),
    "appmetrics": spark.read.format("delta").load("Tables/dbo/bronze_appmetrics"),
    "diagnostic_audit": spark.read.format("delta").load("Tables/dbo/bronze_diagnostic_audit"),
    "platform_metrics": spark.read.format("delta").load("Tables/dbo/bronze_platform_metrics"),
}

for source_name, source_df in bronze_tables.items():
    print(f"Bronze {source_name}: {source_df.count()} rows")


## 2. Normalize FOCUS Costs

Project the source export into the stable cost contract required by Gold. Column lookup is case-insensitive and supports common FOCUS naming variants without inventing values for absent fields.


In [ ]:
df_bronze_costs = bronze_tables["costs"]

df_silver_costs = (
    df_bronze_costs
    .select(
        F.lower(source_column(df_bronze_costs, "ResourceId")).alias("resource_id"),
        F.to_date(source_column(df_bronze_costs, "ChargePeriodStart")).alias("cost_date"),
        source_column(df_bronze_costs, "EffectiveCost", "BilledCost", data_type="double").alias("cost_amount"),
        source_column(df_bronze_costs, "ServiceName", "ServiceCategory").alias("service_name"),
        source_column(df_bronze_costs, "ResourceName", "ServiceName").alias("capacity_name"),
        source_column(df_bronze_costs, "RegionName", "RegionId", "Location").alias("region"),
        source_column(df_bronze_costs, "ResourceGroupName", "ResourceGroup").alias("resource_group"),
        source_column(df_bronze_costs, "ChargeCategory", "ChargeType").alias("charge_category"),
        source_column(df_bronze_costs, "BillingCurrency", "BillingCurrencyCode").alias("billing_currency"),
        source_json_column(df_bronze_costs, "Tags").alias("tags"),
        F.col("_ingestion_timestamp"),
        F.col("_source_file"),
    )
    .filter(F.col("cost_date").isNotNull() & F.col("cost_amount").isNotNull())
    .dropDuplicates(["resource_id", "cost_date", "service_name", "cost_amount", "_source_file"])
    .withColumn("_silver_timestamp", F.current_timestamp())
)

print(f"Silver cost records: {df_silver_costs.count()}")


In [ ]:
df_silver_costs.groupBy("service_name", "billing_currency").agg(
    F.sum("cost_amount").alias("total_cost"),
    F.count("*").alias("line_items"),
).orderBy(F.desc("total_cost")).show(20, truncate=False)


## 3. Conform Operational Events

Standardize application requests, external dependencies, and Azure audit records into one operational event contract. Source-specific details remain available in `properties` and the Bronze tables.


In [ ]:
def normalize_operation(df, operation_type):
    success = source_column(df, "Success", data_type="boolean")
    status_code = source_column(df, "ResultCode", "statusCode", data_type="integer")
    source_severity = F.lower(source_column(df, "SeverityLevel", "Level"))

    return (
        df.select(
            F.to_timestamp(source_column(df, "TimeGenerated", "time")).alias("event_timestamp"),
            source_column(df, "AppRoleName", "ResourceProvider", "category").alias("service_name"),
            source_column(df, "Name", "operationName").alias("operation_name"),
            F.lit(operation_type).alias("operation_type"),
            status_code.alias("status_code"),
            source_column(df, "DurationMs", "durationMs", data_type="double").alias("latency_ms"),
            success.alias("success"),
            source_column(df, "OperationId", "correlationId").alias("operation_id"),
            source_column(df, "ParentId").alias("parent_id"),
            source_column(df, "Url", "Target", "resourceId").alias("target"),
            source_column(df, "Category", "category").alias("category"),
            source_json_column(df, "Properties", "properties").alias("properties"),
            F.col("_ingestion_timestamp"),
            F.col("_source_file"),
        )
        .withColumn(
            "severity",
            F.when(F.col("success") == F.lit(False), F.lit("Error"))
            .when(F.col("status_code") >= 400, F.lit("Error"))
            .when(source_severity.isin("critical", "error"), F.lit("Error"))
            .otherwise(F.lit("Information")),
        )
        .filter(F.col("event_timestamp").isNotNull())
    )


request_operations = normalize_operation(bronze_tables["apprequests"], "request")
dependency_operations = normalize_operation(bronze_tables["appdependencies"], "dependency")
audit_operations = normalize_operation(bronze_tables["diagnostic_audit"], "audit")


In [ ]:
df_silver_operations = (
    request_operations
    .unionByName(dependency_operations, allowMissingColumns=True)
    .unionByName(audit_operations, allowMissingColumns=True)
    .dropDuplicates(["operation_type", "event_timestamp", "operation_id", "operation_name", "_source_file"])
    .withColumn("event_date", F.to_date("event_timestamp"))
    .withColumn("_silver_timestamp", F.current_timestamp())
)

print(f"Silver operational events: {df_silver_operations.count()}")
df_silver_operations.groupBy("operation_type", "severity").count().show(truncate=False)


## 4. Standardize Metrics at a 5-Minute Grain

Aggregate custom agent metrics and Azure platform metrics independently. They have different semantics and remain separate Silver tables.


In [ ]:
df_bronze_appmetrics = bronze_tables["appmetrics"]
agent_metric_events = df_bronze_appmetrics.select(
    F.to_timestamp(source_column(df_bronze_appmetrics, "TimeGenerated", "time")).alias("event_timestamp"),
    source_column(df_bronze_appmetrics, "AppRoleName").alias("service_name"),
    source_column(df_bronze_appmetrics, "Name", "MetricName").alias("metric_name"),
    source_column(df_bronze_appmetrics, "Sum", "Value", "Average", data_type="double").alias("metric_value"),
    source_column(df_bronze_appmetrics, "ItemCount", "Count", data_type="long").alias("sample_count"),
    source_json_column(df_bronze_appmetrics, "Properties").alias("dimensions"),
).filter(F.col("event_timestamp").isNotNull() & F.col("metric_name").isNotNull())

df_silver_agent_metrics = (
    agent_metric_events
    .groupBy(
        F.window("event_timestamp", "5 minutes").alias("metric_window"),
        "service_name",
        "metric_name",
    )
    .agg(
        F.sum("metric_value").alias("metric_value"),
        F.sum(F.coalesce(F.col("sample_count"), F.lit(1))).alias("sample_count"),
        F.first("dimensions", ignorenulls=True).alias("dimensions"),
    )
    .select(
        F.col("metric_window.start").alias("window_start"),
        F.col("metric_window.end").alias("window_end"),
        F.to_date(F.col("metric_window.start")).alias("metric_date"),
        "service_name",
        "metric_name",
        "metric_value",
        "sample_count",
        "dimensions",
    )
    .withColumn("_silver_timestamp", F.current_timestamp())
)

df_bronze_platform_metrics = bronze_tables["platform_metrics"]
platform_metric_events = df_bronze_platform_metrics.select(
    F.to_timestamp(source_column(df_bronze_platform_metrics, "time", "TimeGenerated")).alias("event_timestamp"),
    F.lower(source_column(df_bronze_platform_metrics, "resourceId", "ResourceId")).alias("resource_id"),
    source_column(df_bronze_platform_metrics, "metricName", "MetricName").alias("metric_name"),
    source_column(df_bronze_platform_metrics, "timeGrain", "TimeGrain").alias("time_grain"),
    source_column(df_bronze_platform_metrics, "average", "Average", data_type="double").alias("average_value"),
    source_column(df_bronze_platform_metrics, "minimum", "Minimum", data_type="double").alias("minimum_value"),
    source_column(df_bronze_platform_metrics, "maximum", "Maximum", data_type="double").alias("maximum_value"),
    source_column(df_bronze_platform_metrics, "total", "Total", data_type="double").alias("total_value"),
    source_column(df_bronze_platform_metrics, "count", "Count", data_type="long").alias("sample_count"),
).filter(F.col("event_timestamp").isNotNull() & F.col("metric_name").isNotNull())

df_silver_platform_metrics = (
    platform_metric_events
    .groupBy(
        F.window("event_timestamp", "5 minutes").alias("metric_window"),
        "resource_id",
        "metric_name",
        "time_grain",
    )
    .agg(
        F.avg("average_value").alias("average_value"),
        F.min("minimum_value").alias("minimum_value"),
        F.max("maximum_value").alias("maximum_value"),
        F.sum("total_value").alias("total_value"),
        F.sum("sample_count").alias("sample_count"),
    )
    .select(
        F.col("metric_window.start").alias("window_start"),
        F.col("metric_window.end").alias("window_end"),
        F.to_date(F.col("metric_window.start")).alias("metric_date"),
        "resource_id",
        "metric_name",
        "time_grain",
        "average_value",
        "minimum_value",
        "maximum_value",
        "total_value",
        "sample_count",
    )
    .withColumn("_silver_timestamp", F.current_timestamp())
)

print(f"Silver agent metric windows: {df_silver_agent_metrics.count()}")
print(f"Silver platform metric windows: {df_silver_platform_metrics.count()}")


## 5. Normalize Resource Metadata

Produce one current resource row per Azure resource. Complex tag and property values are retained as JSON strings, while common governance tags are projected into dedicated columns.


In [ ]:
df_bronze_resources = bronze_tables["resource_metadata"]
resource_tags = source_json_column(df_bronze_resources, "tags", "Tags")

df_silver_resources = (
    df_bronze_resources
    .select(
        F.lower(source_column(df_bronze_resources, "id", "ResourceId")).alias("resource_id"),
        source_column(df_bronze_resources, "name", "ResourceName").alias("resource_name"),
        source_column(df_bronze_resources, "type", "ResourceType").alias("resource_type"),
        source_column(df_bronze_resources, "resourceGroup", "ResourceGroupName").alias("resource_group"),
        source_column(df_bronze_resources, "location", "RegionName", "RegionId").alias("region"),
        source_column(df_bronze_resources, "subscriptionId", "SubscriptionId").alias("subscription_id"),
        resource_tags.alias("tags"),
        source_json_column(df_bronze_resources, "properties", "Properties").alias("properties"),
        F.col("_ingestion_timestamp"),
        F.col("_source_file"),
    )
    .filter(F.col("resource_id").isNotNull())
    .dropDuplicates(["resource_id"])
    .withColumn("environment", F.get_json_object("tags", "$.environment"))
    .withColumn("owner", F.get_json_object("tags", "$.owner"))
    .withColumn("cost_center", F.get_json_object("tags", "$.costCenter"))
    .withColumn("_silver_timestamp", F.current_timestamp())
)

print(f"Silver resources: {df_silver_resources.count()}")


## 6. Validate Silver Contracts

Confirm every Silver product exposes the columns required by downstream consumers. Empty sources are reported as warnings because newly enabled Azure exports do not backfill historical data.


In [ ]:
silver_tables = {
    "silver_costs": df_silver_costs,
    "silver_operations": df_silver_operations,
    "silver_agent_metrics": df_silver_agent_metrics,
    "silver_platform_metrics": df_silver_platform_metrics,
    "silver_resources": df_silver_resources,
}

required_columns = {
    "silver_costs": ["resource_id", "cost_date", "cost_amount", "service_name", "capacity_name", "region"],
    "silver_operations": ["event_timestamp", "service_name", "severity", "status_code", "latency_ms"],
    "silver_agent_metrics": ["window_start", "window_end", "metric_name", "metric_value"],
    "silver_platform_metrics": ["window_start", "window_end", "resource_id", "metric_name"],
    "silver_resources": ["resource_id", "resource_name", "resource_type", "resource_group", "region", "tags"],
}

for table_name, table_df in silver_tables.items():
    missing_columns = sorted(set(required_columns[table_name]) - set(table_df.columns))
    assert not missing_columns, f"{table_name} is missing required columns: {missing_columns}"
    row_count = table_df.count()
    status = "PASS" if row_count > 0 else "WARN"
    print(f"[{status}] {table_name}: {row_count} rows")


## 7. Write Silver Delta Tables

Persist each validated Silver product as a Delta table. Overwrite mode keeps workshop reruns deterministic while retaining replayable Bronze data.


In [ ]:
for table_name, table_df in silver_tables.items():
    table_path = f"Tables/dbo/{table_name}"
    print(f"Writing {table_name} to {table_path}...")
    (
        table_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(table_path)
    )
    written_count = spark.read.format("delta").load(table_path).count()
    print(f"  {table_name}: {written_count} rows written successfully.")

print("\nSilver layer transformation complete.")
